# Phase 1 — Data Pipeline & Evaluation Metrics
**Project:** Can LoRA-Adapted SAM Match Specialist Polyp Networks?

This notebook covers the full Phase 1 deliverables:
1. Install dependencies
2. Download all five polyp datasets (PraNet standard packages)
3. Verify train/seen/unseen splits
4. Visualise sample images from each dataset
5. Smoke-test all six evaluation metrics on real images

**GPU needed:** None for this notebook — T4 is fine if you have one, CPU works too.

## 1. Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    !git clone --quiet https://github.com/palism1/msu2026summer_final_project.git
    %cd msu2026summer_final_project
    !pip install -q -r requirements.txt
else:
    print('Running locally — make sure you are in the repo root.')

import os, sys
repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

DATA_ROOT = Path('data/polyp')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Data root: {DATA_ROOT.resolve()}')

## 2. Download Datasets

PraNet releases **two separate Google Drive packages** — one for training data, one for test data.
Both are downloaded automatically below using `gdown`.

| Package | Size | Contains |
|---|---|---|
| TrainDataset | 399.5 MB | Kvasir (900) + CVC-ClinicDB (550) |
| TestDataset  | 327.2 MB | Kvasir (100) + CVC-ClinicDB (62) + CVC-ColonDB (380) + ETIS-LaribPolypDB (196) + CVC-300 (60) |

In [ ]:
import gdown, shutil

# File IDs from PraNet GitHub README (github.com/DengPingFan/PraNet)
TRAIN_ID = '1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb'   # TrainDataset 399.5 MB
TEST_ID  = '1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao'   # TestDataset  327.2 MB

def download_and_extract(file_id, zip_name, dest_dir):
    zip_path = f'/tmp/{zip_name}'
    folder_name = zip_name.replace('.zip', '')
    target = DATA_ROOT / folder_name

    if target.exists():
        print(f'{folder_name}: already downloaded ({len(list(target.rglob("*.jpg")) + list(target.rglob("*.png")))} files)')
        return

    print(f'Downloading {folder_name}...')
    gdown.download(id=file_id, output=zip_path, quiet=False)

    print(f'Extracting...')
    # Unzip into a temp dir, then move into DATA_ROOT
    tmp_extract = Path(f'/tmp/extract_{folder_name}')
    tmp_extract.mkdir(exist_ok=True)
    import zipfile
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(tmp_extract)

    # Find the extracted folder (might be nested)
    extracted = list(tmp_extract.iterdir())
    if len(extracted) == 1 and extracted[0].is_dir():
        shutil.move(str(extracted[0]), str(DATA_ROOT / folder_name))
    else:
        (DATA_ROOT / folder_name).mkdir(exist_ok=True)
        for item in extracted:
            shutil.move(str(item), str(DATA_ROOT / folder_name / item.name))

    print(f'Done: {DATA_ROOT / folder_name}')

download_and_extract(TRAIN_ID, 'TrainDataset.zip', DATA_ROOT)
download_and_extract(TEST_ID,  'TestDataset.zip',  DATA_ROOT)

In [ ]:
# Verify folder structure
expected = {
    'TrainDataset/Kvasir':           900,
    'TrainDataset/CVC-ClinicDB':     612,
    'TestDataset/Kvasir':            100,
    'TestDataset/CVC-ClinicDB':       62,
    'TestDataset/CVC-ColonDB':       380,
    'TestDataset/ETIS-LaribPolypDB': 196,
    'TestDataset/CVC-300':            60,
}

print(f'{"Dataset folder":<34} {"Expected":>10} {"Found":>10} {"OK":>5}')
print('-' * 65)
all_ok = True
for rel_path, expected_count in expected.items():
    folder = DATA_ROOT / rel_path / 'images'
    found = len(list(folder.glob('*'))) if folder.exists() else 0
    ok = '✓' if found == expected_count else ('~' if found > 0 else '✗')
    if ok == '✗':
        all_ok = False
    print(f'{rel_path:<34} {expected_count:>10} {found:>10} {ok:>5}')

print()
print('All datasets ready!' if all_ok else 'Some datasets missing — check the download cell.')

## 3. Verify PraNet Splits

In [ ]:
from src.data import build_splits
from src.data.dataset import verify_splits

try:
    splits = build_splits(DATA_ROOT, seed=42)
    verify_splits(splits)
except FileNotFoundError as e:
    print(f'Missing dataset: {e}')
    splits = None

In [ ]:
# Confirm zero overlap between train and any test split
if splits:
    train_names = {p.name for p in splits['train']['image_paths']}
    for key in ['seen_kvasir', 'seen_clinicdb', 'cvc_colondb', 'etis_larib', 'cvc_300']:
        if key not in splits:
            continue
        test_names = {p.name for p in splits[key]['image_paths']}
        overlap = train_names & test_names
        status = '✓ No overlap' if not overlap else f'✗ OVERLAP: {len(overlap)} files!'
        print(f'{key:<20}: {status}')

## 4. Visualise Samples

In [ ]:
def show_samples(image_paths, mask_paths, title, n=4):
    n = min(n, len(image_paths))
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for i in range(n):
        img = np.array(Image.open(image_paths[i]).convert('RGB'))
        msk = np.array(Image.open(mask_paths[i]).convert('L'))
        axes[0, i].imshow(img);  axes[0, i].set_title(f'Image {i+1}', fontsize=9); axes[0, i].axis('off')
        axes[1, i].imshow(msk, cmap='gray'); axes[1, i].set_title(f'Mask {i+1}', fontsize=9); axes[1, i].axis('off')
    plt.tight_layout(); plt.show()

dataset_dirs = {
    'Kvasir-SEG (train)':        DATA_ROOT / 'TrainDataset' / 'Kvasir',
    'CVC-ClinicDB (train)':      DATA_ROOT / 'TrainDataset' / 'CVC-ClinicDB',
    'CVC-ColonDB (unseen)':      DATA_ROOT / 'TestDataset'  / 'CVC-ColonDB',
    'ETIS-LaribPolypDB (unseen)':DATA_ROOT / 'TestDataset'  / 'ETIS-LaribPolypDB',
    'CVC-300 (unseen)':          DATA_ROOT / 'TestDataset'  / 'CVC-300',
}

for name, folder in dataset_dirs.items():
    if not (folder / 'images').exists():
        print(f'  {name}: not yet downloaded')
        continue
    imgs = sorted((folder / 'images').glob('*'))
    msks = sorted((folder / 'masks').glob('*'))
    show_samples(imgs, msks, name, n=4)

## 5. Augmentation Preview

In [ ]:
from src.data import get_train_transform

transform = get_train_transform(img_size=352)
kvasir_imgs = sorted((DATA_ROOT / 'TrainDataset' / 'Kvasir' / 'images').glob('*'))
kvasir_msks = sorted((DATA_ROOT / 'TrainDataset' / 'Kvasir' / 'masks').glob('*'))

raw_img = np.array(Image.open(kvasir_imgs[0]).convert('RGB'))
raw_msk = np.array(Image.open(kvasir_msks[0]).convert('L'))
raw_msk_f = (raw_msk > 127).astype(np.float32)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Augmentation preview (same image, 6 random versions)', fontsize=13)

axes[0,0].imshow(raw_img);           axes[0,0].set_title('Original'); axes[0,0].axis('off')
axes[1,0].imshow(raw_msk, cmap='gray'); axes[1,0].set_title('GT mask'); axes[1,0].axis('off')

for i in range(1, 6):
    out = transform(image=raw_img, mask=raw_msk_f)
    img_t = out['image'].permute(1,2,0).numpy()
    img_display = np.clip(img_t * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406]), 0, 1)
    axes[0,i].imshow(img_display); axes[0,i].set_title(f'Aug {i}'); axes[0,i].axis('off')
    axes[1,i].imshow(out['mask'].numpy(), cmap='gray'); axes[1,i].set_title(f'Mask {i}'); axes[1,i].axis('off')

plt.tight_layout(); plt.show()

## 6. Evaluation Metrics — Smoke Test

All six PraNet metrics on a real GT mask vs three prediction types.

In [ ]:
from src.metrics import dice_score, iou_score, mae_score, weighted_f_measure, s_measure, e_measure

gt = (np.array(Image.open(kvasir_msks[0]).convert('L')) > 127).astype(np.float32)
rng = np.random.default_rng(0)

cases = {
    'Perfect  (pred = gt)':  gt,
    'Random   (uniform)':    rng.uniform(0, 1, gt.shape).astype(np.float32),
    'All zeros (no polyp)':  np.zeros_like(gt),
}

print(f'{"Case":<24} {"Dice":>7} {"IoU":>7} {"MAE":>7} {"wFm":>7} {"Sm":>7} {"Em":>7}')
print('-' * 72)
for name, pred in cases.items():
    pred_bin = (pred >= 0.5).astype(np.float32)
    print(f'{name:<24}'
          f' {dice_score(pred_bin, gt):>7.4f}'
          f' {iou_score(pred_bin, gt):>7.4f}'
          f' {mae_score(pred, gt):>7.4f}'
          f' {weighted_f_measure(pred_bin, gt):>7.4f}'
          f' {s_measure(pred, gt):>7.4f}'
          f' {e_measure(pred, gt):>7.4f}')

In [ ]:
from src.metrics import MetricTracker

tracker = MetricTracker()
for i in range(10):
    gt_i = (np.array(Image.open(kvasir_msks[i]).convert('L')) > 127).astype(np.float32)
    pred_i = rng.uniform(0, 1, gt_i.shape).astype(np.float32)
    tracker.update(pred_i, gt_i)

print('MetricTracker over 10 images (random predictions):')
print(' ', tracker.summary())

## 7. Dataset Statistics

In [ ]:
sizes = []
for msk_path in kvasir_msks:
    m = np.array(Image.open(msk_path).convert('L'))
    sizes.append(100 * (m > 127).sum() / m.size)

plt.figure(figsize=(8, 4))
plt.hist(sizes, bins=30, edgecolor='black', color='steelblue', alpha=0.8)
plt.xlabel('Polyp area (% of image)'); plt.ylabel('Count')
plt.title('Kvasir-SEG — polyp size distribution (n=1000)')
plt.axvline(np.mean(sizes), color='red', linestyle='--', label=f'Mean={np.mean(sizes):.1f}%')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Mean={np.mean(sizes):.1f}%  Median={np.median(sizes):.1f}%  Std={np.std(sizes):.1f}%')

## Phase 1 Summary

| Deliverable | Status |
|---|---|
| Repository & dependencies | ✅ |
| All 5 datasets downloaded (PraNet packages) | ✅ |
| Train/seen/unseen split builder | ✅ (`src/data/dataset.py`) |
| Zero train/test overlap verified | ✅ |
| All 6 evaluation metrics | ✅ (`src/metrics/segmentation.py`) |
| Augmentation pipeline | ✅ (`src/data/transforms.py`) |

**Next:** `02_unet_baseline.ipynb` — train the U-Net specialist baseline (T4 GPU, ~1-2 hrs).